# bathy2strat


## Bathymetry Formatter
Read in netcdf files and compile into bathymetry cube

In [2]:
from bathymetry_analysis import process_bathy_formatter, plot_bathy_formatter_outputs

# Define paths
nc_dir = r"C:\surf\300_Data\320_BogueInlet\netcdf\UTM_renamed"
out_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\data"
plot_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\plots\python"

# Step 1: Process bathymetry (read, clean, interpolate, save outputs)
# Moderate shrink test: tighter than default, but avoids collapsing to blank outputs.
# output_format options: "netcdf" (CF-compliant, default), "mat" (MATLAB struct), "both"
bathy_result = process_bathy_formatter(
    nc_dir=nc_dir,
    out_dir=out_dir,
    dx=20.0,
    drop_year=2008,
    interp_method="linear",
    boundary_tightness=2.0,
    overlap_erosion_cells=1,
    output_format="netcdf",  # Change to "mat" or "both" as needed
)

bathy = bathy_result.bathy
print(f"Processing complete. Regridded bathymetry shape: {bathy.z.shape}")

Processing complete. Regridded bathymetry shape: (93, 92, 24)


# Plot output
Plot 

In [4]:
# Step 2: Generate QC plots from either in-memory result or saved file path
# Choose one source below:
plot_source = bathy_result
# plot_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
# plot_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

plot_bathy_formatter_outputs(
    process_result=plot_source,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extents_cmap_name="viridis",
    extent_boundary_method="mask",
    boundary_tightness=2.0,
    overlap_erosion_cells=1,
    mask_nan_plots=True,
)

bathy = bathy_result.bathy
print(f"Plotting complete. Survey dates (MATLAB datenum): {bathy.t.flatten()}")
print(f"Grid spacing: x={bathy.x[0, 1] - bathy.x[0, 0]:.1f}m, y={bathy.y[1, 0] - bathy.y[0, 0]:.1f}m")

Plotting complete. Survey dates (MATLAB datenum): [732313. 733194. 733774. 734929. 735447. 735690. 736116. 736269. 736482.
 736604. 736786. 736969. 737212. 737365. 737485. 737730. 737943. 738126.
 738308. 738491. 738673. 738826. 739068. 739221.]
Grid spacing: x=20.0m, y=20.0m


In [8]:
# Utility: pick one source for all downstream analysis
# 1) in-memory process result
analysis_source = bathy_result
# 2) saved netCDF
# analysis_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
# 3) saved MAT
# analysis_source = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

import importlib
import bathy_formatter as bf

# Reload in case this module was imported before recent edits.
bf = importlib.reload(bf)

bathy_analysis = bf.load_for_analysis(analysis_source)
print(f"Analysis source type: {type(analysis_source).__name__}")
print(f"Analysis bathy shape (y, x, t): {bathy_analysis.z.shape}")

Analysis source type: BathyProcessResult
Analysis bathy shape (y, x, t): (93, 92, 24)


In [1]:
from bathymetry_analysis import plot_bathy_formatter_outputs

plot_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\plots\python"
out_dir = r"C:\surf\500_Analysis\522_CarolinaInlets\data"

nc_path = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.nc"
mat_path = out_dir + r"\BogueInlet_2005-2023_bathyReGrid_2023s_dx20m_py.mat"

# Test plotting directly from .nc path
plot_bathy_formatter_outputs(
    process_result=nc_path,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extent_boundary_method="convex",
    mask_nan_plots=True,
)

# Test plotting directly from .mat path
plot_bathy_formatter_outputs(
    process_result=mat_path,
    plot_dir=plot_dir,
    bathy_cmap_name="kg2",
    extent_boundary_method="convex",
    mask_nan_plots=True,
)

print("Path-based plotting test complete for both .nc and .mat")

Loaded from file path: skipping raw extents/raw survey plots (source surveys unavailable).
Loaded from file path: skipping raw extents/raw survey plots (source surveys unavailable).
Path-based plotting test complete for both .nc and .mat


## Deposit Age Calculator
Calculate the age of deposits.